In [3]:
import gc
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc, torch
import torchvision.transforms as transforms
from IPython.display import display
from torch.utils.data import DataLoader

from codes.sprint.models import *
from codes.sprint.training import *
from codes.sprint.data import *
from codes.sprint.utils import *


In [4]:
# import gc
# import os
# import sys
# sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "codes"))
# from pathlib import Path
# 
# os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# 
# import numpy as np
# import pandas as pd
# import scanpy as sc
# import torch
# import torchvision.transforms as transforms
# from IPython.display import display
# from torch.utils.data import DataLoader
# 
# from sprint.models import *
# from sprint.training import *
# from sprint.data import *
# from sprint.utils import *
# 
# device = get_device()
# set_seed(42)
# print(f"Using device: {device}")


In [5]:
os.getcwd()

'/media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github'

In [7]:
path_human = "datas/breast/Breast_Human_Final.h5ad"
path_mouse = "datas/breast/Breast_Mouse_Final.h5ad"
if not (os.path.exists(path_human) and os.path.exists(path_mouse)):
    raise FileNotFoundError("Breast h5ad files not found")

In [8]:
common_transform = transforms.Compose([transforms.Resize((32, 32)), transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

In [9]:
ad_h = sc.read_h5ad(path_human, backed="r")
ad_m = sc.read_h5ad(path_mouse, backed="r")
common_genes = sorted(set(g.upper() for g in ad_h.var_names).intersection(g.upper() for g in ad_m.var_names))
p_h = [str(x) for x in list(ad_h.uns.get("protein_names", []))]
p_m = [str(x) for x in list(ad_m.uns.get("protein_names", []))]
common_proteins = sorted(set(p_h).intersection(p_m))
if len(common_proteins) == 0 and p_h and p_m:
    p_h_upper = {p.upper(): p for p in p_h}
    p_m_upper = {p.upper(): p for p in p_m}
    common_proteins = [p_h_upper[u] for u in sorted(set(p_h_upper).intersection(p_m_upper))]
del ad_h, ad_m
gc.collect()

912

In [10]:
train_dataset = BreastMultimodalDataset(path_human, target_genes=common_genes, target_proteins=common_proteins or None, transform=common_transform)
val_dataset = BreastMultimodalDataset(path_mouse, target_genes=common_genes, target_proteins=common_proteins or None, transform=common_transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
NUM_GENES = train_dataset.rna_data.shape[1]
NUM_PROTEINS = train_dataset.protein_data.shape[1]
print(f"Final Config: {NUM_GENES} genes, {NUM_PROTEINS} proteins")

Loading data from: Breast_Human_Final.h5ad ...
Loading data from: Breast_Mouse_Final.h5ad ...
Final Config: 15462 genes, 11 proteins


In [12]:
model_name = 'C2-20260611bz256'
MODEL_REGISTRY = {
    model_name: Model_C2,
}

MAIN_MODELS = [
    (MODEL_REGISTRY[model_name], model_name),
]

ABLATION_MODELS = [
]

In [13]:
model_kwargs = {"num_genes": NUM_GENES}
models_to_train = MAIN_MODELS
all_histories = {}

In [10]:
for i, (model_cls, name) in enumerate(models_to_train):
    print(f"\n[{i + 1}/{len(models_to_train)}] Training {name}")
    all_histories[name] = train_engine(model_cls, name, train_loader, val_loader, NUM_PROTEINS, device, epochs=50,lr=0.01, model_kwargs=model_kwargs)


[1/1] Training C2-20260611bz256
C2-20260611bz256: resume=False, ignoring existing checkpoint models_breast/C2-20260611bz256/ckpt.pth


[C2-20260611bz256] Ep 1/50:   0%|          | 0/17 [00:00<?, ?it/s]

C2-20260611bz256 epoch 1: loss=276.9570, val_pcc=0.0322, val_rmse=65587.9141, lr=1.00e-02


[C2-20260611bz256] Ep 2/50:   0%|          | 0/17 [00:00<?, ?it/s]

C2-20260611bz256 epoch 2: loss=124.9224, val_pcc=-0.0792, val_rmse=31.8636, lr=1.00e-02


...

C2-20260611bz256 epoch 48: loss=0.4680, val_pcc=0.3107, val_rmse=2.2665, lr=3.87e-03


[C2-20260611bz256] Ep 49/50:   0%|          | 0/17 [00:00<?, ?it/s]

C2-20260611bz256 epoch 49: loss=0.4868, val_pcc=0.3190, val_rmse=2.3009, lr=3.87e-03


[C2-20260611bz256] Ep 50/50:   0%|          | 0/17 [00:00<?, ?it/s]

C2-20260611bz256 epoch 50: loss=0.3753, val_pcc=0.3142, val_rmse=2.1384, lr=3.87e-03, lr_decay=3.87e-03->3.49e-03
